# Routing BENCHMARK — Gemma 4 E4B chess-coach (for the report)

Runs the **serious** routing ablation on Kaggle (T4, up to 12h — no Colab disconnect), three conditions over a large stratified sample of the validation set:

| condition | weights | harness contract | isolates |
|---|---|---|---|
| **A. adapter + harness** | trained v4 LoRA | yes | the product |
| **B. base + harness** | base (LoRA off) | yes | what the **SFT weights** bought |
| **C. base, no harness** | base (LoRA off) | **no** | what the **harness contract** bought |

For each: confusion matrix + precision/recall/F1 + exact-name + format validity + throughput, then the layer deltas (A vs B = SFT weights, B vs C = harness, A vs C = full system). B and C reuse the SAME loaded model with the LoRA disabled (no 2nd load, no OOM); no fabricated external baselines — honest and reproducible from our own artifacts.

**Setup:** Settings → Accelerator → **GPU T4 x2** (one is enough). Add **HF_TOKEN** under Add-ons → Secrets. Run top to bottom; the report + PNGs land in `/kaggle/working/` and render inline.

In [ ]:
# Cell 1 - config
import os
BASE_REPO = 'unsloth/gemma-4-E4B-it'   # the base the adapter was trained on
REPO_URL  = 'https://github.com/RyanDev1st/llm-and-engine.git'
BRANCH    = 'feat/chess-coach-sft'
ADAPTER_REPO = 'RyanDev1st/gemma4-chesscoach-ckpt-v4'
TAG = 'best'
WORKDIR   = '/kaggle/working/llm-and-engine'
BASE_DIR  = '/kaggle/working/gemma4_e4b'
ADAPTER_DIR = '/kaggle/working/adapter/best'
# THREE conditions run sequentially, so size for ~12h total. At ~10-14s/row:
#   PER_SLICE=25 ~= 750 rows -> ~3h/condition -> ~9h for all three (fits with headroom).
#   PER_SLICE=0 (full 2731-row val) is ~33h total -> use the time budget + accept a partial last condition.
PER_SLICE   = 25          # rows/slice (0 = full val). 25 is statistically strong and fits 3 conditions.
TIME_BUDGET = 3 * 3600    # seconds PER condition (hard stop; writes the matrix from completed rows)
print(f'per_slice={PER_SLICE} (0=full val) | budget/condition {TIME_BUDGET//3600}h | 3 conditions => ~{3*TIME_BUDGET//3600}h max')

In [ ]:
# Cell 2 - clone repo (prints HEAD so the commit is visible) + deps
import subprocess, sys
env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
if not os.path.exists(WORKDIR):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(['git','-C',WORKDIR,'fetch','origin',BRANCH], check=True, env=env)
    subprocess.run(['git','-C',WORKDIR,'reset','--hard',f'origin/{BRANCH}'], check=True, env=env)
print('HEAD:', subprocess.run(['git','-C',WORKDIR,'log','-1','--oneline'], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','peft','bitsandbytes','sentencepiece','protobuf','python-chess'], check=True)
os.environ['PYTHONPATH'] = f'{WORKDIR}/src/llm'
import transformers, peft, bitsandbytes
print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)

In [ ]:
# Cell 3 - download the E4B base (~9GB) + the v4 adapter
from huggingface_hub import login, snapshot_download
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret('HF_TOKEN'))
except Exception:
    login(os.environ['HF_TOKEN'])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=['*.json','*.safetensors','*.model','*.txt','tokenizer*'])
snapshot_download(repo_id=ADAPTER_REPO, local_dir='/kaggle/working/adapter', allow_patterns=[f'{TAG}/*'])
need = ('adapter_model.safetensors', 'adapter_config.json')
assert all(os.path.exists(f'{ADAPTER_DIR}/{f}') for f in need), f'adapter not at {ADAPTER_DIR}'
print('base + adapter ready:', os.listdir(ADAPTER_DIR))

In [ ]:
# Cell 4 - RUN THE BENCHMARK (adapter vs base; hours-safe; writes the report + PNGs)
# Loads the model ONCE; the base condition reuses it with the LoRA disabled (no 2nd load, no OOM).
# Saves docs/findings/<date>-routing-benchmark.md + -adapter.png + -base.png; copied to /kaggle/working.
import os, sys, shutil, subprocess, glob
os.environ['CHESS_HF_BASE'] = BASE_DIR
cmd = [sys.executable, '-m', 'llm_training.eval_benchmark', '--adapter', ADAPTER_DIR,
       '--per-slice', str(PER_SLICE), '--time-budget', str(TIME_BUDGET)]
print('running:', ' '.join(cmd), '\n', flush=True)
subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env={**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}, check=True)
# surface the artifacts in the Kaggle output pane + show the matrices inline
for f in glob.glob(f'{WORKDIR}/docs/findings/*-routing-benchmark*'):
    shutil.copy(f, '/kaggle/working/')
from IPython.display import Image, Markdown, display
md = sorted(glob.glob('/kaggle/working/*-routing-benchmark.md'))
if md: display(Markdown(open(md[-1]).read()))
for png in sorted(glob.glob('/kaggle/working/*-routing-benchmark-*.png')):
    print(png); display(Image(filename=png))